# LeKiwi single-shot auto-setup — from box to calibrated robot

This notebook is both a **walk-through** (read it top to bottom to understand the
method) and an **executable setup tool**: run it end-to-end on the robot's
Raspberry Pi 5 and it takes a brand-new LeKiwi (SO-101 arm + 3 omni wheels,
Feetech STS3215 servos) to a validated, LeRobot-ready calibration.

**Everything is deterministic.** Every decision is a numeric gate with an
explicit threshold (see the `GATES` dict below) — no human or LLM judgment is
needed mid-run, so the notebook can run *blind*:

```bash
pip install nbclient ipykernel        # one-time, in the robot venv
jupyter execute lekiwi_setup.ipynb    # headless single-shot run
```

A failed **hard gate** stops the run with torque relaxed (safe). A failed
**soft gate** falls back (e.g. a suspect sweep is refused and the seed value is
kept). **Every failure prints its likely cause and the fix** — re-fold, clear
space, charge — so a blind run always ends with actionable instructions.

### What you need
- The LeKiwi powered on, on the floor, with **at least 60 cm of clear space in
  every direction** (the calibration choreography reaches 49 cm from the base
  centre — a nearby wall *will* be hit and flagged as an obstacle).
- **Battery charged** — a 12 V pack should read ≥ 11 V. At ~9.6 V the servos
  still respond but can't lift the arm against gravity, and the sweeps stall
  into false stops (L-009).
- The servo bus on `/dev/ttyACM0` (override with `LEKIWI_PORT`).
- This notebook next to the `lekiwi_*.py` / `calib_*.py` scripts and the cached
  `calib_choreography_lekiwi.json` (shipped together). MuJoCo + the Ekumen model
  assets are **optional** — with them the choreography is re-derived and
  re-verified locally; without them the sim-verified cache is used.

### The single human step
**Before running**: fold the arm into the *seed pose* — each joint pushed gently
to its hard stop as shown in §4 (a picture is rendered there if sim is
available). That is the only physical intervention; the notebook verifies you
did it correctly (§5.2) and aborts safely if not.

> **Model note:** the bundled collision model is the open-source Ekumen *SO-100* LeKiwi; it is geometrically valid for the *SO-101* too (identical kinematics, same gripper reach — see the FAQ). Set `LEKIWI_FINGER_MARGIN` for your floor surface.
>
> Provenance: every protection in this notebook exists because something went
> wrong on real hardware first. The gotcha boxes (**L-001 … L-010**) quote the
> incident ledger in the incident ledger.

> **See also — the two senses of "calibration."** This notebook does *encoder
> bring-up*: it maps each joint's raw servo **ticks → radians** (offset, sign,
> usable range) on the physical robot. That is distinct from *parameter
> system-identification* — recovering the arm's physical **damping** by
> differentiable simulation. For that method, worked end-to-end on the same
> SO-101 arm in sim, see
> [`examples/calibrate_so101.ipynb`](../examples/calibrate_so101.ipynb).

## 0 · Vocabulary and pipeline

Six terms, used precisely throughout:

| Term | Question it answers | Concrete artifact here |
|---|---|---|
| **Simulation** | What does the world model say is safe/reachable? | MuJoCo collision model (real meshes, derived floor, allowed-collision matrix) |
| **Choreography** | What exact motions should the hardware play? | home pose + per-joint stances + sweep arcs (`calib_choreography_lekiwi.json`) |
| **Calibration** | What is each joint's encoder→angle map? | `q = (tick − offset) / (sign · TICKS_PER_RAD)` + usable range |
| **Registration** | Is that map expressed in the *planning model's* frame? | sign/zero agreement with the MJCF/URDF joint conventions |
| **Verification** | Did the mechanism do what we *commanded*? (process vs spec) | arrival checks, realized-vs-commanded stance checks |
| **Validation** | Does the result match *physical reality*? (outcome vs ground truth) | full-range sweeps to both hard stops, scale vs datasheet |

```
SIMULATION → CHOREOGRAPHY → [hardware] → CALIBRATION → REGISTRATION → VALIDATION
 (world model)  (the plan)               (the map)     (map ↔ model)   (map ↔ reality)
                                          └── VERIFICATION runs throughout ──┘
```

> **L-006 — why registration is a separate phase, *before* validation.** An
> internally-consistent calibration (clean symmetric sweep, valid LeRobot file)
> can still be mis-registered to the model frame: a flipped sign/zero leaves a
> *symmetric* single-joint sweep unchanged — only multi-joint poses expose it.
> On our robot this made a sim-safe elbow sweep drive the gripper into the
> floor while every commanded tick was "correct". Calibrated ≠ registered:
> two separate gates.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))  # the lekiwi_*.py scripts live next to this notebook

PORT = os.environ.get("LEKIWI_PORT", "/dev/ttyACM0")
ASSETS = Path(
    os.environ.get(
        "LEKIWI_ASSETS", "/tmp/lekiwi_ek/packages/lekiwi_sim/lekiwi_sim/assets"
    )
)
CHOREO_PATH = Path(
    os.environ.get("LEKIWI_CHOREO", str(HERE / "calib_choreography_lekiwi.json"))
)
OUT = Path(os.environ.get("LEKIWI_OUT", str(HERE / "out")))
OUT.mkdir(exist_ok=True)

RUN_HW = Path(PORT).exists()  # robot connected?
try:
    import mujoco  # noqa: F401

    RUN_SIM = ASSETS.exists()  # model assets available?
except ImportError:
    RUN_SIM = False
RENDER = False  # optional: re-render seed/dry-run media (needs sim)
CHECK_VISION = False  # optional: deterministic camera cross-check (needs cams)

# Every numeric decision in one place. The run is deterministic: each gate
# compares a measurement against these and passes or fails — nothing else.
GATES = {
    "voltage_v": (9.0, 13.0),  # 12 V STS3215 variant; NOTE: >=11 V strongly
    # recommended — at ~9.6 V gravity-loaded
    # sweeps stall into false stops (L-009)
    "temp_max_c": 45.0,  # servo temperature ceiling at start
    "steady_ticks": 3,  # encoder jitter ceiling when parked
    "nudge_ticks": 140,  # sign-probe excursion (~12 deg)
    "nudge_free_frac": 0.5,  # >= this fraction moved -> direction is free
    "nudge_block_frac": 0.25,  # <= this fraction moved -> direction blocked
    "excursion_rad": 0.5,  # registration test angle from the seed stop
    "scale_rtol": 0.12,  # |measured/datasheet - 1| tolerance
    "grip_scale_rtol": 0.30,  # looser for the gripper (springy stops)
    "anchor_tol_ticks": 25,  # hard-stop re-anchor repeatability
    "arrive_tol_ticks": 130,  # pose arrival
    "stance_tol_ticks": 50,  # realized-vs-commanded stance (closed loop)
    "floor_clearance_m": 0.08,  # sag + floor covering + transit lag budget
    # (L-005/L-010)
    "coverage_min": 0.985,  # required sweep coverage in sim
}

#: What to tell the human when a gate fails: likely cause + how to fix, then
#: re-run the notebook. Matched by longest gate-name prefix.
REMEDIES = {
    "hw.all_servos": "a servo is missing from the bus - check power and re-seat "
    "the daisy-chain cables, then re-run",
    "hw.voltage": "battery out of range - CHARGE the pack (12 V system should "
    "read >= 11 V for reliable sweeps), then re-run",
    "hw.temperature": "a servo is hot - let the robot cool down, then re-run",
    "seed.readings_steady": "the robot moved during the read - keep hands off "
    "and the floor still, then re-run",
    "seed.at_stop": "this joint is NOT folded against its stop - re-fold the "
    "arm to match out/seed_pose.png (gripper UP, every joint pushed to its "
    "stop), then re-run",
    "cal.sign_vs_inference": "informational only - the motion probe overrides "
    "the read-only inference; no action needed",
    "cal.lerobot_in_range": "exported offset out of register range - report "
    "this as a bug (should be impossible after wrap+clamp)",
    "reg.excursion": "the joint did not move as the model predicts - most "
    "likely the seed fold used the WRONG end for this joint, or it rests "
    "on a false stop (L-007): re-fold exactly per the picture and re-run",
    "reg.anchor": "the hard stop did not repeat - check for debris or an "
    "obstacle at this joint's stop, then re-run",
    "reg.multi_joint_home": "a joint stalled on the way to home - an OBSTACLE "
    "is in the workspace (wall/object within 60 cm?) or the battery is "
    "weak: clear >= 60 cm around the robot, charge, re-fold the seed pose, "
    "re-run",
    "reg.home_realized": "the home pose did not realize - obstacle contact or "
    "weak battery: clear >= 60 cm around the robot, charge, re-fold, re-run",
    "transit.torque_margin": "torque saturated mid-move - the arm pushed "
    "against something (wall, floor) or the battery is weak: check "
    "the robot for contact, clear >= 60 cm, charge, re-fold, re-run",
    "val.sweep": "false stop mid-sweep (obstacle or weak battery) - the seed "
    "value was KEPT so the calibration is still safe; clear space, charge "
    "to >= 11 V, and re-run to validate this joint",
    "hw.temp_ok": "a servo is overheating mid-run (overload) - power off, let it cool, and re-run; if it persists the choreography is loading that joint too hard",
    "grip.": "gripper anomaly - check nothing is lodged between the fingers "
    "and that the jaw moves freely by hand, then re-run",
}

RESULTS = {"gates": []}
_CLEANUP = []  # callables run before a hard abort (relax torque, close bus)


def _remedy(name: str):
    best = ""
    for k in REMEDIES:
        if name.startswith(k) and len(k) > len(best):
            best = k
    return REMEDIES.get(best)


def gate(name: str, ok: bool, detail: str = "", hard: bool = True) -> bool:
    """Record a pass/fail decision; a failed *hard* gate aborts the run safely.

    Every failure prints a human-readable likely cause + fix (REMEDIES), so a
    blind run always ends with actionable instructions, not a stack trace."""
    rem = None if ok else _remedy(name)
    RESULTS["gates"].append(
        {"gate": name, "ok": bool(ok), "hard": hard, "detail": detail, "remedy": rem}
    )
    print(("  PASS  " if ok else "  FAIL  ") + f"{name:36s} {detail}")
    if not ok and rem:
        print(f"         -> {rem}")
    if not ok and hard:
        for fn in _CLEANUP:
            fn()
        raise RuntimeError(
            f"hard gate failed: {name} - {detail}\nFIX: {rem or 'see ledger'}"
        )
    return ok


print(f"hardware : {'YES (' + PORT + ')' if RUN_HW else 'no - sim-only walk-through'}")
print(
    f"sim      : {'YES (' + str(ASSETS) + ')' if RUN_SIM else 'no - cached choreography'}"
)
print(f"outputs  : {OUT}")

## 1 · Simulation — build an *honest* world model

Before any motor moves, we reason in a MuJoCo model of this exact embodiment
(the open-source Ekumen LeKiwi: real meshes, the mobile base with its
electronics stack). Two fidelity rules, both learned on hardware:

> **L-005a — derive the floor, never hard-code it.** Our first scene placed the
> floor plane 10 cm below the model origin — phantom air the planner happily
> used. The model itself knows where the floor is: the lowest resting wheel/base
> geometry. `LekiwiCollisionModel` derives it (≈ −0.4 cm) and snaps the plane
> there.

> **L-005b — require a clearance *margin*, not just "no contact".** Real arms
> sag under load and calibration residuals shift poses. `is_safe()` therefore
> demands the whole arm stay `floor_clearance` (5 cm) above the derived floor.

In [ ]:
if RUN_SIM:
    from calib_planner_lekiwi import LekiwiCollisionModel

    cm = LekiwiCollisionModel(ASSETS, floor_clearance=GATES["floor_clearance_m"])
    print(f"model: {cm.model.njnt} joints, {cm.model.ngeom} geoms")
    print(f"derived real floor z = {cm._floor_z * 100:+.2f} cm (not hard-coded; L-005)")
    for j, (lo, hi) in cm.jnt_range.items():
        print(f"  {j:12s} range [{lo:+.3f}, {hi:+.3f}] rad")
    gate("sim.model_loads", True, f"floor z={cm._floor_z:+.4f} m")
else:
    print("SKIPPED (no mujoco/assets) - the sim-verified cached choreography is used")

## 2 · Choreography — the sim-verified motion plan

The plan the hardware will play: a compact collision-free **home** the arm
returns to between steps, and a **per-joint stance** from which that one joint
can sweep its *entire* range to both stops. Every transit and every sweep is
verified collision-free in sim, and stances are chosen to *maximize* floor
clearance (sag headroom) among full-coverage candidates.

> **L-001 — never run a naive all-joints sweep.** Our very first touch-off run
> calibrated joints while the others sat in arbitrary poses; the gripper jammed
> between base and floor and had to be e-stopped. All motion must come from an
> a-priori per-embodiment plan.

> **L-004 — a joint blocked from one pose is not blocked, period.** We wrongly
> concluded two joints "physically cannot reach the far stop" because every
> sweep started from one shared pose. Searching a *tailored stance per joint*
> proved 100% coverage for all five. Never assume reachability from one pose.

In [ ]:
import subprocess

if CHOREO_PATH.exists():
    choreo = json.loads(CHOREO_PATH.read_text())
    print(f"loaded cached choreography: {CHOREO_PATH.name}")
elif RUN_SIM:
    # deterministic derivation (the script uses a fixed RNG seed)
    subprocess.run(
        [sys.executable, str(HERE / "calib_choreography_lekiwi.py")],
        check=True,
        cwd=str(HERE.parent),
    )
    derived = HERE.parent / "examples/media/calib_choreography_lekiwi.json"
    choreo = json.loads(derived.read_text())
    CHOREO_PATH.write_text(json.dumps(choreo, indent=2) + "\n")  # cache for the Pi
else:
    raise RuntimeError("no cached choreography and no sim to derive one")

gate(
    "choreo.sim_verified",
    bool(choreo.get("verified")),
    "full sequence (home -> stance -> sweep -> home) collision-free in sim",
)
gate(
    "choreo.floor_margin",
    choreo.get("floor_clearance_m", 0.0) >= GATES["floor_clearance_m"],
    f"plan enforced {choreo.get('floor_clearance_m')} m above floor "
    f"z={choreo.get('floor_z_m')} m",
)

print("\nhome:", {k: round(v, 2) for k, v in choreo["home"].items()})
print(f"{'joint':14s} {'coverage':>8s} {'floor clearance':>16s}")
for s in choreo["steps"]:
    if s.get("feasible"):
        clr = s.get("min_floor_clearance_m")
        print(
            f"{s['joint']:14s} {s['coverage'] * 100:7.1f}% "
            f"{(clr * 100 if clr is not None else float('nan')):15.1f} cm"
        )
        gate(
            f"choreo.coverage.{s['joint']}",
            s["coverage"] >= GATES["coverage_min"],
            f"{s['coverage']:.1%}",
        )

## 3 · Hardware bring-up — discover and health-check the bus

Nine STS3215 servos on one serial bus: arm IDs 1–6, wheels 7–9.

Gotchas baked into `Bus`: the STS3215 speaks **`protocol_end=0`**
(little-endian words — `protocol_end=1` reads garbage positions), baud is
1 000 000, and every motion command caps the **torque-limit register first** so
any unexpected contact is a gentle press, not an impact.

Wheel *identity* (which servo drives which physical wheel) cannot be verified
without watching the robot — we trust the assembly convention
(7 = front-left, 8 = back, 9 = front-right; verified once on this unit) and the
appendix points at `lekiwi_wheels.py` for the interactive check.

In [ ]:
import lekiwi_autosetup as las

bus = None
if RUN_HW:
    SPECS = las.LAYOUTS["lekiwi"]
    IDS = tuple(s.servo_id for s in SPECS)
    ARM = [s for s in SPECS if s.kind == "arm" and s.name not in las.NO_SEED_JOINTS]
    bus = las.Bus(PORT, las.DEFAULT_BAUD)  # protocol_end=0: STS3215
    _CLEANUP.append(lambda: (bus.relax_all(IDS), bus.close()))

    found = bus.scan(IDS)
    gate(
        "hw.all_servos",
        set(found) == set(IDS),
        f"found {sorted(found)}, expected {sorted(IDS)}",
    )
    volts = {s.name: bus.voltage(s.servo_id) for s in SPECS}
    temps = {s.name: bus.temperature(s.servo_id) for s in SPECS}
    lo_v, hi_v = GATES["voltage_v"]
    gate(
        "hw.voltage",
        all(lo_v <= v <= hi_v for v in volts.values()),
        f"{min(volts.values()):.1f}-{max(volts.values()):.1f} V (window {lo_v}-{hi_v})",
    )
    gate(
        "hw.temperature",
        all(t <= GATES["temp_max_c"] for t in temps.values()),
        f"max {max(temps.values()):.0f} C",
    )
    bus.relax_all(IDS)  # limp and safe; the human may still be near the robot
else:
    print("SKIPPED (no hardware) - cells below print what they WOULD do")

## 4 · Seed pose — the one human step

Fold the arm so **every joint rests against a hard stop** — push each joint
gently in the direction below *until it stops*. No "middle" positions, no
eyeballing: a hard stop is unambiguous, repeatable, and is the only physical
landmark a novice can hit exactly.

Why this works: at a stop, the joint is at a *known model angle* (the URDF
limit). One known (tick, angle) pair anchors the offset — without moving a
single motor.

| joint | push toward | rests at (model angle) |
|---|---|---|
| shoulder_pan | base spun fully counter-clockwise (from above) | upper limit +1.92 rad |
| shoulder_lift | upper arm folded fully toward vertical/forward | upper limit +1.75 rad |
| elbow_flex | forearm folded fully closed, **gripper pointing UP** | lower limit −1.69 rad |
| wrist_flex | wrist pitched fully back toward the forearm | lower limit −1.66 rad |
| wrist_roll | roll turned fully to its stop | upper limit +2.84 rad |
| gripper | leave as-is (soft stop — not seeded) | — |

> **These end labels are *measured*, not assumed.** The fold's ends were
> determined on hardware by a camera-anchored probe (nudge each joint toward
> its interior, compare observed image motion against the model's prediction).
> An earlier hand-written version of this table had three ends mislabeled — and
> every proprioceptive check passed anyway, because mirroring shoulder_pan and
> shoulder_lift *together* produces a nearly identical-looking fold (~39° of
> yaw apart). Photos, encoders, and span-based validation are all blind to it.
> If you re-derive this table for a new embodiment, measure it (L-008).

> **L-007 — the elbow direction is the one trap.** Both elbow folds "look
> folded", but only gripper-UP reaches the elbow's true mechanical stop. In the
> gripper-DOWN fold the jaw rests against the base/shoulder *before* the elbow
> reaches its limit — a firm, convincing, **false** landmark at an unknown
> angle. (The stall probe cannot tell a true stop from firm link contact; only
> validation catches it later.) Match the picture: gripper UP.

If sim is available the next cell renders this pose to `out/seed_pose.png`.
For a blind run, set the pose **before** launching (or set
`LEKIWI_SEED_WAIT_S=120` to get a window after launch). §5.2 *verifies* every
joint really is at *a* stop and aborts safely if not — the notebook never
trusts the human.

In [ ]:
#: which hard stop the fold puts each joint against. MEASURED on hardware via
#: the camera-anchored end probe (see the L-008 note above) — do not hand-edit
#: for a new embodiment without re-measuring.
SEED_STOP = {
    "shoulder_pan": "upper",
    "shoulder_lift": "upper",
    "elbow_flex": "lower",
    "wrist_flex": "lower",
    "wrist_roll": "upper",
}
#: layout joint -> Ekumen model joint
LAY2MODEL = {
    "shoulder_pan": "Rotation",
    "shoulder_lift": "Pitch",
    "elbow_flex": "Elbow",
    "wrist_flex": "Wrist_Pitch",
    "wrist_roll": "Wrist_Roll",
}


def seed_angle(spec: las.JointSpec) -> float:
    lo, hi = spec.limits
    return lo if SEED_STOP[spec.name] == "lower" else hi


if RUN_SIM:
    cm.set_q(
        {
            LAY2MODEL[s.name]: seed_angle(s)
            for s in las.LAYOUTS["lekiwi"]
            if s.name in SEED_STOP
        }
    )
    cam = mujoco.MjvCamera()
    cam.lookat[:] = [0.0, 0.0, 0.12]
    cam.distance, cam.azimuth, cam.elevation = 0.65, 150, -22
    r = mujoco.Renderer(cm.model, 480, 640)
    r.update_scene(cm.data, cam)
    import PIL.Image

    PIL.Image.fromarray(r.render()).save(OUT / "seed_pose.png")
    print(f"seed pose rendered -> {OUT / 'seed_pose.png'}")
    from IPython.display import Image, display

    display(Image(str(OUT / "seed_pose.png")))

wait_s = int(os.environ.get("LEKIWI_SEED_WAIT_S", "0"))
if RUN_HW and wait_s:
    print(f"waiting {wait_s}s for the operator to set the seed pose ...")
    time.sleep(wait_s)

if RUN_HW:
    # read-only steadiness check: a parked, untouched arm reads a stable encoder
    jitter = {}
    for s in ARM:
        reads = [bus.position(s.servo_id) for _ in range(5)]
        jitter[s.name] = max(reads) - min(reads)
    gate(
        "seed.readings_steady",
        all(j <= GATES["steady_ticks"] for j in jitter.values()),
        f"max encoder jitter {max(jitter.values())} ticks (robot must be still)",
    )

## 5 · Calibration — building the encoder→angle map

The map per joint is `tick = offset + sign · TICKS_PER_RAD · q`. We build it in
four sub-steps, cheapest and safest first:

1. **Offset** (§5.1) — read-only, from the seed stop.
2. **Sign** (§5.2) — tiny motion: a stall-probe at the stop. Also *verifies* the
   seed pose.
3. **Range** (§5.3) — derived: model limits pushed through the fitted map.
4. **Export** (§5.4) — the LeRobot-format file, overflow-safe.

(*Scale* — measured ticks-per-rad vs the datasheet's 651.9 — needs both stops,
so it is measured during **validation** (§7), not here.)

> **L-002 — encoder wrap and the which-stop trap.** `shoulder_pan`'s travel
> crosses the encoder's 0/4095 boundary, so one of its stops lies *beyond* the
> encoder span. Early on we anchored it to the wrong stop — a clean-looking
> ~130° calibration error. Two defenses below: `calibrate_from_seed` infers the
> stop that keeps the range representable, and the sign probe (§5.2)
> independently confirms which side the physical stop is on. All offset
> arithmetic is modular (wrap-aware).

In [ ]:
# 5.1 - offsets from the seed stops (read-only: cannot collide)
if RUN_HW:
    seed_cal, notes = {}, {}
    for s in ARM:
        cal, note = las.calibrate_from_seed(bus, s)
        seed_cal[s.name], notes[s.name] = cal, note
        flag = f"  ! {note}" if note else ""
        print(
            f"  {s.name:14s} offset={cal['offset_ticks']:7.1f}  "
            f"inferred stop q={cal['seed_stop_rad']:+.2f}{flag}"
        )
    gate(
        "cal.offsets_inferred",
        len(seed_cal) == len(ARM),
        f"{len(seed_cal)} arm joints anchored, zero motor motion",
    )

### 5.2 · Sign — the stall-probe (camera-free and deterministic)

An encoder alone can never tell you the **sign** (which tick direction is +q):
both hypotheses fit any single joint's readings equally well. You need one bit
of external physical truth. The seed pose provides it for free:

*At a hard stop, exactly one direction is blocked.* Nudge the servo ±140 ticks
(load-capped, gentle): the direction that moves freely points to the range
interior; the blocked one presses into the stop. Since §4 tells us *which*
limit the joint rests at, the free tick-direction fixes the sign:

- at the **lower** limit, interior = +q → `sign = free_dir`
- at the **upper** limit, interior = −q → `sign = −free_dir`

The same probe **verifies the seed pose** (a joint not at a stop moves freely
both ways → hard abort) and re-anchors the offset with the now-known sign.

*(Historical note: our first robot got its signs from a fully-autonomous
two-camera inference — still available as an optional cross-check in §6 — but
the stall-probe needs no camera at all and is the default for blind runs.)*

In [ ]:
def wrap_diff(a: int, b: int, res: int = las.RESOLUTION) -> int:
    """Shortest signed tick distance a-b on the circular encoder (L-002)."""
    return ((a - b + res // 2) % res) - res // 2


def probe_from_stop(sid: int) -> tuple[int | None, int]:
    """At a hard stop: which tick direction is free? -> (dir or None, anchor).

    Classification is *relative*, not absolute (L-012): a compliant stop lets the
    joint rebound off it when released, so the blocked direction still travels a
    little before re-contacting. A fixed "blocked < 35 ticks" test fails on such
    joints (the springy wrist rebounds ~44 ticks). Instead a stop shows one
    direction clearly free (moved most of the nudge) AND the other distinctly
    less (<= half as far). Both directions free -> not at a stop."""
    t0 = bus.position(sid)
    progress = {}
    for d in (+1, -1):
        goal = max(0, min(las.RESOLUTION - 1, t0 + d * GATES["nudge_ticks"]))
        bus.start_move(
            sid, goal, speed=las.TOUCH_OFF_SPEED, max_load=las.TOUCH_OFF_MAX_LOAD
        )
        time.sleep(1.0)
        progress[d] = d * wrap_diff(bus.position(sid), t0)  # ticks toward goal
        bus.start_move(
            sid, t0, speed=las.TOUCH_OFF_SPEED, max_load=las.TOUCH_OFF_MAX_LOAD
        )  # back to the stop
        time.sleep(1.0)
    bus.set_torque(sid, False)
    anchor = bus.position(sid)
    pp, pm = progress[+1], progress[-1]
    hi, lo = max(pp, pm), min(pp, pm)
    # one direction clearly free AND the other distinctly more constrained
    at_stop = hi >= GATES["nudge_free_frac"] * GATES["nudge_ticks"] and lo <= 0.5 * hi
    free = (+1 if pp > pm else -1) if at_stop else None
    return free, anchor


if RUN_HW:
    for s in ARM:
        q_stop = seed_angle(s)
        free, anchor = probe_from_stop(s.servo_id)
        gate(
            f"seed.at_stop.{s.name}",
            free is not None,
            "one free + one blocked direction"
            if free is not None
            else "joint is NOT against a stop - re-do the seed pose",
        )
        sign = free if SEED_STOP[s.name] == "lower" else -free
        off = (anchor - sign * las.TICKS_PER_RAD * q_stop) % las.RESOLUTION
        # cross-check the read-only which-stop inference of 5.1 (soft):
        agree = (
            abs(wrap_diff(round(off), round(seed_cal[s.name]["offset_ticks"])))
            <= GATES["anchor_tol_ticks"]
            and sign == seed_cal[s.name]["sign"]
        )
        gate(
            f"cal.sign_vs_inference.{s.name}",
            agree,
            f"sign {sign:+d}, offset {off:.1f} "
            + ("(agrees with 5.1)" if agree else "(5.1 disagreed - probe wins)"),
            hard=False,
        )
        lo, hi = s.limits
        edges = sorted(off + sign * las.TICKS_PER_RAD * q for q in (lo, hi))
        seed_cal[s.name].update(
            sign=sign,
            offset_ticks=off,
            range_min_ticks=int(max(0, min(las.RESOLUTION - 1, round(edges[0])))),
            range_max_ticks=int(max(0, min(las.RESOLUTION - 1, round(edges[1])))),
            method="seed_stop_plus_probe",
        )
        if edges[0] < 0 or edges[1] > las.RESOLUTION - 1:
            print(
                f"    note: {s.name} far stop is beyond the encoder span - "
                "range clamped, excluded from full sweeps (L-002)"
            )

In [ ]:
# 5.3 - usable ranges: model limits pushed through the fitted map
if RUN_HW:
    SWEEPABLE = []
    print(f"{'joint':14s} {'sign':>4s} {'offset':>8s} {'range (ticks)':>15s}  notes")
    for s in ARM:
        c = seed_cal[s.name]
        lo, hi = s.limits
        raw = sorted(
            c["offset_ticks"] + c["sign"] * las.TICKS_PER_RAD * q for q in (lo, hi)
        )
        wraps = raw[0] < 0 or raw[1] > las.RESOLUTION - 1
        if not wraps:
            SWEEPABLE.append(s.name)
        print(
            f"{s.name:14s} {c['sign']:+4d} {c['offset_ticks']:8.1f} "
            f"[{c['range_min_ticks']:4d}..{c['range_max_ticks']:4d}]  "
            + ("wraps encoder - sweep excluded" if wraps else "full sweep OK")
        )
    gate("cal.ranges_derived", len(SWEEPABLE) >= 3, f"sweepable joints: {SWEEPABLE}")

### 5.4 · Export — overflow-safe LeRobot format

> **L-003 — the homing-offset overflow.** LeRobot writes `homing_offset` into a
> *signed 11-bit* servo register (±2047). The naive `2048 − mid` overflows when
> the encoder zero is off-centre (LeRobot issue #3193, "Magnitude exceeds
> 2047"). For a single-turn encoder the correction is modular, so
> `lerobot_entry` *wraps* then clamps — and we gate that every exported value is
> register-writable.

In [ ]:
if RUN_HW:
    for s in (x for x in SPECS if x.kind == "wheel"):
        seed_cal[s.name] = las.wheel_zero(bus, s)  # read-only zero for each wheel
    lr = {n: las.lerobot_entry(c) for n, c in seed_cal.items()}
    gate(
        "cal.lerobot_in_range",
        all(-2047 <= e["homing_offset"] <= 2047 for e in lr.values()),
        "all homing offsets register-writable (L-003)",
    )
    (OUT / "calibration_seed.json").write_text(
        json.dumps(
            {
                "schema": "lekiwi-autosetup/notebook-v1",
                "robot": "lekiwi",
                "stage": "seed (pre-validation)",
                "calibrations": list(seed_cal.values()),
            },
            indent=2,
            default=float,
        )
        + "\n"
    )
    print(f"wrote {OUT / 'calibration_seed.json'} (gripper: soft stop, not seeded)")

## 6 · Registration — is the map in the *model's* frame?

The map is now internally consistent. That is **not** the same as being
registered to the planning model (see L-006 in §0) — and the choreography is
expressed in model angles, so we must close this gap *before* trusting any
multi-joint motion. Two deterministic gates, in increasing exposure:

1. **Per-joint excursion** — from the seed stop, command +0.5 rad *into the
   interior through the map* and compare the achieved tick displacement against
   the prediction. A mirrored sign moves the wrong way (`got · pred < 0`); a
   bad scale misses the magnitude; both fail loudly at a pose we know is safe
   (the probe just proved the interior is free). Then return to the stop and
   check the anchor repeats within tolerance.
2. **Multi-joint home** — pose the choreography's compact home through the map
   with the closed loop (arrival check, stiff hold, realized-vs-commanded
   verification). This is the cheapest pose that exercises *all* joints
   simultaneously — the configuration class where mis-registration shows up.

Joints whose excursion path would cross the encoder boundary (shoulder_pan)
skip gate 1 and rely on gate 2 (L-002).

In [ ]:
if RUN_HW:
    for s in ARM:
        c = seed_cal[s.name]
        q_stop = seed_angle(s)
        interior = +1 if SEED_STOP[s.name] == "lower" else -1
        anchor = bus.position(s.servo_id)
        dtick = round(c["sign"] * las.TICKS_PER_RAD * interior * GATES["excursion_rad"])
        target = anchor + dtick
        if not 0 <= target <= las.RESOLUTION - 1:
            print(
                f"  {s.name:14s} SKIPPED - excursion crosses the encoder "
                "boundary; covered by the multi-joint gate (L-002)"
            )
            continue
        bus.start_move(s.servo_id, target, speed=las.TOUCH_OFF_SPEED, max_load=0.35)
        time.sleep(1.6)
        got = wrap_diff(bus.position(s.servo_id), anchor)
        rel = abs(got - dtick) / max(1, abs(dtick))
        gate(
            f"reg.excursion.{s.name}",
            got * dtick > 0 and rel <= GATES["scale_rtol"],
            f"predicted {dtick:+d} ticks, achieved {got:+d} (err {rel:.0%})",
        )
        bus.start_move(
            s.servo_id,
            anchor,
            speed=las.TOUCH_OFF_SPEED,
            max_load=las.TOUCH_OFF_MAX_LOAD,
        )
        time.sleep(1.6)
        drift = abs(wrap_diff(bus.position(s.servo_id), anchor))
        bus.set_torque(s.servo_id, False)
        gate(
            f"reg.anchor.{s.name}",
            drift <= GATES["anchor_tol_ticks"],
            f"stop re-anchor drift {drift} ticks",
        )

In [ ]:
if RUN_HW:
    import lekiwi_calibrate_full as lcf

    def transit_check(label: str) -> None:
        """Flag torque saturation seen DURING the last move (L-010) AND say why
        (L-011): a depleted pack sags hard under load, a mechanical obstacle does
        not. `classify_stall` reads that from the bus — no model needed onboard."""
        names = {lcf.ID[j]: j for j in lcf.ARM}
        sat = [names.get(s, s) for s in lcf.LAST_MOVE["saturated"]]
        if sat:
            cause, why = lcf.classify_stall()
            gate(
                f"transit.torque_margin.{label}",
                False,
                f"saturated on {sat} - {cause}: {why}",
                hard=False,
            )

    cal_map = {
        n: {"offset": c["offset_ticks"], "sign": c["sign"]}
        for n, c in seed_cal.items()
        if n in lcf.ID
    }
    home_t = {
        lcf.ID[j]: lcf.to_tick(cal_map, j, choreo["home"][lcf.LAY2MODEL[j]])
        for j in lcf.ARM
    }
    print("first multi-joint motion: compact home, closed-loop ...")
    ok = lcf.move_to(bus, home_t)
    transit_check("to_home")
    gate(
        "reg.multi_joint_home",
        ok,
        "all joints arrived (simultaneous move, stall-aborting)",
    )
    # stiff hold AT TARGET: holding at the present position would freeze any
    # gravity shortfall from the gentle approach (arrived within ARRIVE_TOL but
    # outside STANCE_TOL) — re-commanding the target under a rigid cap closes it
    lcf.hold_at(bus, home_t)
    miss = lcf._held_deviation(bus, home_t)
    gate(
        "reg.home_realized",
        not miss,
        "realized pose == commanded pose" if not miss else f"deviations {miss}",
    )

In [ ]:
# OPTIONAL deterministic camera cross-check (off by default; needs cameras).
# An eye-in-hand wrist camera removes the one ambiguity markerless external
# vision can't (a global sign flip): optical-flow *curl* during a wrist_roll
# nudge is unambiguous in the camera's own frame. See lekiwi_vision_sign.py
# (external cam, relative signs) and vision_sign_solve.py (model correlation).
if CHECK_VISION and RUN_HW:
    print(
        "see lekiwi_vision_sign.py / vision_sign_solve.py - deterministic CV, "
        "no LLM in the loop"
    )
else:
    print(
        "SKIPPED (CHECK_VISION=False) - the stall-probe already fixed signs "
        "without cameras"
    )

### 6½ · Gripper — the one soft-stop joint

The gripper was excluded from seeding because the whole method anchors on
*hard* stops and the gripper has none worth trusting: its open side is
compliant, and its closed "stop" is the fingers pressing together — springy,
with a contact angle that depends on force. Skipping it entirely, though,
leaves the LeRobot file without a gripper entry, so it gets its own gentler
protocol here (the arm is parked at the held home, so gripper motion moves
nothing else):

1. **Touch both ends twice** at two torque caps (0.15 then 0.30) — very gentle,
   the fingers are plastic.
2. **Identify which end is the fingers** by *compliance*: the finger-contact
   end deflects measurably further when pressed harder; the open mechanical
   stop barely moves. That one physical bit resolves the open/close direction
   without a camera (the mirror lesson of L-008, solved by springiness instead
   of vision).
3. **Anchor the map** at the gentle-touch ticks against the URDF limits, with a
   loose scale gate (springy stops make the span sloppy — ±30%).
4. **Functional test**: command open and near-closed through the new map and
   verify arrival — the only joint that gets actively position-tested here,
   because it's the only one whose stops were too soft to trust passively.

In [ ]:
if RUN_HW:
    gspec = next(s for s in SPECS if s.name == "gripper")
    glo, ghi = gspec.limits
    gid = gspec.servo_id

    # touch both ends at a gentle then a firmer cap; deflection difference
    # identifies the springy finger-contact (closed) end
    soft, hard_t, found = {}, {}, {}
    for d in (-1, +1):
        soft[d], found[d] = las._drive_to_stop(bus, gid, d, max_load=0.15)
        hard_t[d], _ = las._drive_to_stop(bus, gid, d, max_load=0.30)
    bus.set_torque(gid, False)
    if not gate(
        "grip.both_ends",
        found[-1] and found[+1],
        f"gentle ends at {soft[-1]} / {soft[+1]}",
        hard=False,
    ):
        print("    gripper skipped - no entry will be exported")
    else:
        defl = {d: abs(hard_t[d] - soft[d]) for d in (-1, +1)}
        closed_dir = max(defl, key=defl.get)
        gate(
            "grip.closed_end_identified",
            abs(defl[-1] - defl[+1]) >= 4,
            f"compliance deflection -1:{defl[-1]} +1:{defl[+1]} ticks "
            f"-> fingers at dir {closed_dir:+d}",
            hard=False,
        )
        t_closed, t_open = soft[closed_dir], soft[-closed_dir]
        gsign = +1 if t_open > t_closed else -1
        off = 0.5 * (
            (t_closed - gsign * las.TICKS_PER_RAD * glo)
            + (t_open - gsign * las.TICKS_PER_RAD * ghi)
        )
        m_tpr = abs(t_open - t_closed) / (ghi - glo)
        gate(
            "grip.span_vs_model",
            abs(m_tpr / las.TICKS_PER_RAD - 1) <= GATES["grip_scale_rtol"],
            f"measured {m_tpr:.0f} ticks/rad vs datasheet {las.TICKS_PER_RAD:.0f}",
            hard=False,
        )
        edges = sorted(off + gsign * las.TICKS_PER_RAD * q for q in (glo, ghi))
        clamp = lambda x: int(max(0, min(las.RESOLUTION - 1, round(x))))  # noqa: E731
        seed_cal["gripper"] = {
            "joint": "gripper",
            "servo_id": gid,
            "sign": gsign,
            "ticks_per_rad": las.TICKS_PER_RAD,
            "offset_ticks": off,
            "range_min_ticks": clamp(edges[0]),
            "range_max_ticks": clamp(edges[1]),
            "measured_ticks_per_rad": m_tpr,
            "method": "gripper_touch_compliance",
        }

        # functional test: drive open then near-closed through the new map.
        # Close direction is reliable; opening from a firmly-pressed-closed jaw
        # can bind under a gentle cap, so the open arrival tolerance is loose and
        # this is a soft, informational gate (it does not block the calibration).
        def g_tick(q):
            return clamp(off + gsign * las.TICKS_PER_RAD * q)

        track = {}
        for label, q, tol in (("open", 1.2, 200), ("near-closed", 0.15, 60)):
            bus.start_move(gid, g_tick(q), speed=400, max_load=0.30)
            time.sleep(1.3)
            track[label] = abs(bus.position(gid) - g_tick(q))
            print(
                f"    {label:12s} q={q:+.2f} -> tick {g_tick(q)}  "
                f"arrival err {track[label]} ticks (tol {tol})"
            )
        gate(
            "grip.tracks",
            track["near-closed"] <= 60 and track["open"] <= 200,
            f"open err {track['open']}, close err {track['near-closed']} ticks",
            hard=False,
        )

        # L-011: park the jaw CLOSED and HOLD it (gentle) through all the arm
        # motion below, so the end-effector geometry matches the model's default
        # and an open jaw can't splay the fingertips into the floor mid-sweep.
        # park FULLY closed by driving to the measured closed stop: q=0.05
        # left the jaw ~18% open, splaying the fingers into the floor (L-014e)
        cpos, _ = las._drive_to_stop(bus, gid, closed_dir, max_load=0.30)
        GRIP_CLOSED = cpos
        bus.start_move(gid, GRIP_CLOSED, speed=300, max_load=0.45)  # firm hold
        time.sleep(0.8)
        print(
            f"    parked FULLY CLOSED at tick {bus.position(gid)} "
            "(re-asserted before every sweep)"
        )

## 7 · Validation — full-range sweeps against physical reality

The seed calibration is anchored at *one* stop per joint. Validation drives
each joint across its **entire** range to **both** stops from its sim-verified
stance, which checks the map against ground truth and upgrades it:

- **both stops found** → the measured span gives `measured_ticks_per_rad`;
  gate it against the datasheet (±12%). A big miss means a false stop —
  something blocked the joint before its mechanical limit.
- **midpoint anchor** → with two stops, the offset becomes the true mechanical
  centre — strictly better than the single-stop seed anchor, so clean sweeps
  are *adopted* into the final calibration.
- **suspect refusal** → a sweep whose stance drifted, whose scale misses the
  gate, or that found fewer than two stops is **refused**: the seed value is
  kept and the failure is reported. Bad data never ships.

Protections from the incident ledger, all active here: stances were chosen for
maximum floor headroom (L-005), held joints get a near-rigid hold and the
stance is verified *before* releasing the swept joint and re-checked *after*
(L-005/L-006), every move is load-capped and stall-aborting (L-001).

In [ ]:
if RUN_HW:
    steps = {st["joint"]: st for st in choreo["steps"] if st.get("feasible")}
    sweeps = {}
    for joint in [j for j in SWEEPABLE if j in steps]:
        spec = next(s for s in SPECS if s.name == joint)
        if "gripper" in seed_cal:  # keep the jaw firmly closed (L-011): an open
            gid = seed_cal["gripper"]["servo_id"]  # jaw splays fingers into the floor
            bus.start_move(gid, GRIP_CLOSED, speed=300, max_load=0.40)
        hot = lcf.hottest(
            bus, [s.servo_id for s in SPECS]
        )  # overload/heat sentinel (L-014b)
        gate(
            "hw.temp_ok",
            hot[1] < lcf.TEMP_ABORT_C,
            f"warmest servo {hot[0]} at {hot[1]:.0f} C",
        )
        print(f"\n=== {joint}: pose stance -> full both-stop sweep ===")
        held = lcf.pose_stance(bus, cal_map, joint, steps[joint]["stance"])
        transit_check(f"stance_{joint}")
        if held is None:
            sweeps[joint] = {"adopted": False, "why": "stance not realized"}
            gate(
                f"val.sweep.{joint}",
                False,
                "stance not realized - refused, seed kept",
                hard=False,
            )
            continue
        v_rest = bus.voltage(spec.servo_id)
        bus.set_torque(spec.servo_id, False)
        # held joints are watched DURING the sweep (L-013): a sag aborts before
        # the gripper can reach the floor, rather than being seen only afterward
        r, abort = lcf.guarded_touch_off(bus, spec, held)
        if abort is not None:
            sweeps[joint] = {"adopted": False, "why": f"sweep aborted: {abort}"}
            gate(
                f"val.sweep.{joint}",
                False,
                f"ABORTED mid-sweep ({abort}) - no floor contact; seed kept",
                hard=False,
            )
            lcf.move_to(bus, home_t)
            lcf.hold_at(bus, home_t)
            continue
        v_load = bus.voltage(spec.servo_id)
        post = lcf._held_deviation(bus, held)
        m = r.get("measured_ticks_per_rad")
        err = abs(m / las.TICKS_PER_RAD - 1) if m else None
        clean = (
            r.get("stops_found") == 2
            and not post
            and err is not None
            and err <= GATES["scale_rtol"]
        )
        # a short span at a sagging voltage = battery; a short span at a steady
        # voltage = obstacle/false stop (L-009 vs L-011), so the remedy is exact
        why = ""
        if not clean:
            why = (
                " [battery: sagged to %.1f V]" % v_load
                if v_rest - v_load >= lcf.VOLT_SAG_BATTERY or v_load < 11.0
                else " [voltage steady %.1f V -> false stop / obstacle]" % v_load
            )
        sweeps[joint] = {
            "adopted": clean,
            "stance_held": not post,
            "scale_rel_err": err,
            "v_rest": v_rest,
            "v_load": v_load,
            **r,
        }
        gate(
            f"val.sweep.{joint}",
            clean,
            f"stops={r.get('stops_found')} stance_held={not post} "
            + (f"scale_err={err:.0%}" if err is not None else "no scale")
            + why,
            hard=False,
        )
        lcf.move_to(bus, home_t)  # back to the verified hub between joints
        transit_check(f"home_after_{joint}")
        lcf.hold_at(bus, home_t)
    (OUT / "validation_sweeps.json").write_text(
        json.dumps(sweeps, indent=2, default=float) + "\n"
    )
    print(f"\nwrote {OUT / 'validation_sweeps.json'}")

In [ ]:
# adopt clean sweeps (midpoint anchor beats single-stop seed); refuse the rest
if RUN_HW:
    final = {n: dict(c) for n, c in seed_cal.items()}
    for j, r in sweeps.items():
        if r.get("adopted"):
            final[j].update(
                offset_ticks=r["offset_ticks"],
                range_min_ticks=r["range_min_ticks"],
                range_max_ticks=r["range_max_ticks"],
                measured_ticks_per_rad=r["measured_ticks_per_rad"],
                method="touch_off_sweep_validated",
            )
    lr = {n: las.lerobot_entry(c) for n, c in final.items()}
    gate(
        "out.lerobot_in_range",
        all(-2047 <= e["homing_offset"] <= 2047 for e in lr.values()),
        "final homing offsets register-writable",
    )
    (OUT / "calibration_final.json").write_text(
        json.dumps(
            {
                "schema": "lekiwi-autosetup/notebook-v1",
                "robot": "lekiwi",
                "method": "seed-stop anchor + stall-probe signs + registration gates "
                "+ choreographed validation sweeps",
                "calibrations": list(final.values()),
            },
            indent=2,
            default=float,
        )
        + "\n"
    )
    (OUT / "lekiwi_lerobot.json").write_text(json.dumps(lr, indent=2) + "\n")
    print(f"{'joint':14s} method")
    for n, c in final.items():
        print(f"{n:14s} {c['method']}")
    print(f"\nwrote {OUT / 'calibration_final.json'} + {OUT / 'lekiwi_lerobot.json'}")

## 8 · Verification summary — the gate ledger

Verification ran *throughout* (every gate above is a verification of one step);
this section is the roll-up. Reminder of the distinction one last time:
**verification** = each commanded step did what the spec said (process);
**validation** = the resulting map matches physical reality (outcome). A run
can verify perfectly and still fail validation — that is exactly how a
registration bug presents (L-006), which is why both ledgers matter.

In [ ]:
print(f"{'GATE':38s} {'KIND':5s} {'RESULT':7s} DETAIL")
for g in RESULTS["gates"]:
    print(
        f"{g['gate']:38s} {'hard' if g['hard'] else 'soft':5s} "
        f"{'PASS' if g['ok'] else 'FAIL':7s} {g['detail']}"
    )

hard_fails = [g for g in RESULTS["gates"] if not g["ok"] and g["hard"]]
soft_fails = [g for g in RESULTS["gates"] if not g["ok"] and not g["hard"]]
(OUT / "setup_report.json").write_text(
    json.dumps(RESULTS, indent=2, default=float) + "\n"
)

if bus is not None:
    # park the arm back in the compact SEED fold (gripper tucked + closed) so
    # it ends clear of the floor, then lower gently before cutting torque
    # (L-014d/e: a limp arm is long enough to drape to the floor; the seed
    # fold is mechanically stable and barely sags when relaxed)
    try:
        seed_t = {
            lcf.ID[s.name]: lcf.to_tick(cal_map, s.name, seed_angle(s)) for s in ARM
        }
        lcf.move_to(bus, seed_t)
        lcf.hold_at(bus, seed_t)
    except Exception as _e:
        print("  park-to-seed skipped:", _e)
    lcf.settle_and_relax(bus, [lcf.ID[j] for j in lcf.ARM])
    bus.relax_all(IDS)
    bus.close()

print(
    f"\n{len(RESULTS['gates'])} gates: "
    f"{sum(g['ok'] for g in RESULTS['gates'])} passed, "
    f"{len(soft_fails)} soft-failed (fallbacks applied), "
    f"{len(hard_fails)} hard-failed"
)

fixable = [
    g
    for g in hard_fails + soft_fails
    if g.get("remedy") and "informational" not in g["remedy"]
]
if fixable:
    print("\nWHAT TO FIX BEFORE THE NEXT RUN:")
    seen = set()
    for g in fixable:
        if g["remedy"] in seen:
            continue
        seen.add(g["remedy"])
        print(f"  [{g['gate']}]")
        print(f"    {g['remedy']}")

verdict = (
    "ABORTED SAFELY"
    if hard_fails
    else "SETUP COMPLETE"
    if RUN_HW
    else "SIM WALK-THROUGH COMPLETE (no hardware connected)"
)
print("\nVERDICT:", verdict)
print(f"report: {OUT / 'setup_report.json'}")

## Appendix — troubleshooting, limits, and what's next

### Symptom → cause → fix (the incident ledger, condensed)

| # | Symptom | Root cause | Defense in this notebook |
|---|---|---|---|
| L-001 | gripper jammed base↔floor on first run | naive sweep, no plan | all motion comes from the sim-verified choreography; load caps + stall aborts everywhere |
| L-002 | ~130° offset error on shoulder_pan | range crosses encoder 0/4095; anchored to the wrong stop | wrap-aware math, which-stop inference + stall-probe cross-check, wrap joints excluded from sweeps |
| L-003 | LeRobot crash "Magnitude exceeds 2047" | homing offset is signed 11-bit | wrap-then-clamp in `lerobot_entry`, gated on export |
| L-004 | "joint can't reach its far stop" (false) | every sweep started from one shared pose | per-joint stance search, 100% coverage proven in sim |
| L-005 | gripper hit the floor mid-sweep | phantom floor (−10 cm hard-coded) + held-joint sag at a soft torque cap | floor derived from the model; clearance budget; clearance-maximizing stances; stiff holds-at-target; realized-stance closed loop |
| L-006 | sweep dove to the floor though sim said clear | calibration internally fine but mis-registered to the model frame | registration phase (§6) *before* any choreography; suspect sweeps refused, seed kept |
| L-007 | elbow "stop" that wasn't its limit | gripper-down fold rests the jaw on the base — a firm false landmark | seed poses interpenetration-checked in sim; gripper-UP called out in §4 |
| L-008 | seed table wrong for 3 joints, every check passed | pan+lift mirror produces a near-identical fold; spans are mirror-invariant | end labels *measured* by a camera-anchored probe, not hand-annotated |
| L-009 | sweeps stalled in place (scale −98 %) | 9.6 V battery → torque-starved against gravity | voltage gate + ≥11 V guidance; refusal logic keeps seed values |
| L-010 | gripper grazed the floor / arm bumped a wall mid-run | transits checked only at endpoints; lag + obstacle invisible | PRESENT_LOAD monitored *during* every move (`transit.torque_margin`); 8 cm floor budget incl. floor covering; ≥60 cm clear-space requirement |

### Known limits (honest list)
- **Gripper** is calibrated by the soft-stop protocol (§6½): compliance
  identifies the finger end, anchors are gentle touches, scale gate is loose
  (±30 %). Good enough for teleop; refine from a teleop range recording if you
  need tighter.
- **Wheel identity** (servo↔physical wheel) needs eyes: run
  `lekiwi_wheels.py spin` once and watch. The default assumes the assembly
  convention 7=front-left, 8=back, 9=front-right.
- **A registration failure is reported, not auto-corrected.** The excursion
  gate tells you *which* joint and *which way* it is wrong; the remedy text
  tells you what to do. Fixing is deliberate, not silent.
- The choreography is embodiment-specific. For a different arm, re-derive it
  (`calib_choreography_lekiwi.py` for LeKiwi; `calib_planner_generic.py`
  transfers the method to arbitrary URDF/MJCF arms) — and re-measure the seed
  end labels (L-008).

### After setup
- Drop `out/lekiwi_lerobot.json` into LeRobot's calibration directory.
- **Drift**: hard stops don't move — re-run §7 (validation) periodically; a
  growing offset delta or scale error is your recalibration signal.
- **Digital twin**: with the map registered (§6 passed), live
  `PRESENT_POSITION` telemetry drives a faithful real-time MuJoCo mirror —
  the natural next build (and the full answer to L-010's transit gap).

### File map
`lekiwi_autosetup.py` (bus, touch-off, seed inference, LeRobot export) ·
`lekiwi_calibrate_full.py` (closed-loop executor primitives, transit monitor) ·
`calib_planner_lekiwi.py` (collision model, derived floor) ·
`calib_choreography_lekiwi.py` (stance search + sim verification) ·
the incident ledger

## FAQ — what it does when things go sideways (and why it was built this way)

**Q. Do I really need all this? LeRobot already has a manual calibration step.**
Manual works — until a novice mis-folds a joint, or the battery is low, or the
arm drifts, and you get a silently-wrong calibration with no signal that
anything's off. This procedure is the manual step made *automated, validated, and
self-diagnosing*: it measures the same numbers, then **proves** they're right and
tells you exactly what to fix when they're not. If you have one robot and a
charged battery and you fold it perfectly, manual is faster today. For a novice, a
fleet, recurring drift, or anyone who wants to *know* it's correct rather than
*hope*, this wins.

**Q. What if I fold the seed pose wrong?** It's caught, and it tells you which
joint. Every joint is probed for "is it actually against a stop?" (a joint folded
to the wrong place moves freely both ways → flagged). Then the registration phase
drives each joint to a known model angle and checks it moves the predicted way —
a mirrored or mis-folded joint fails *there*, before any full-speed motion, naming
the joint and the direction. The infamous failure mode where two joints are
mirrored into a near-identical-looking fold (invisible to photos and encoders) is
why the seed-end labels are *measured*, not hand-trusted.

**Q. It stalled mid-sweep / a joint "couldn't reach its stop."** The run reads the
*reason* off the bus: a depleted pack **sags hard** under load while a mechanical
obstacle does **not** — so the report says either "battery: sagged to 9.8 V →
charge" or "voltage steady → false stop / obstacle." Either way the suspect value
is **refused and the safe seed value is kept**, so nothing wrong ships. Charge or
clear space and re-run just that joint.

**Q. The arm bumped a wall / the charge cable.** The choreography reaches 49 cm
from the base, so it needs **≥60 cm of clear space** — a nearby wall or a tether
*will* be contacted. Every move's torque is monitored *during* the motion (not
just at the endpoint), so a bump is flagged as contact (voltage steady, torque
saturated) rather than silently dragged through.

**Q. Why does it move so slowly and gently?** Every motion is torque-capped, and
where it matters the cap is *computed* from the model's gravity-hold torque for
that exact pose — "only the calculated amount of torque" — so a press into a stop
is a nudge, not an impact, and an unexpected obstacle stalls harmlessly.

**Q. It refused a joint but printed SETUP COMPLETE — is my robot calibrated?**
Yes. A refused *validation sweep* keeps that joint's seed value, which was already
anchored at a real hard stop and confirmed by the registration phase. Validation
is a *bonus* cross-check that upgrades the offset to the true mechanical centre
when it succeeds; when it's refused, you still have a correct, usable calibration —
just without that one extra confirmation.

**Q. How can it run "blind," with no one watching?** Every decision is a numeric
gate against an explicit threshold (the `GATES` dict) — no human or model judgment
mid-run. A hard gate aborts safely (torque off); a soft gate falls back and is
logged. The run ends with a deduplicated **"WHAT TO FIX"** list, so a headless
`jupyter execute` finishes with instructions, not a stack trace.

**Q. What about edge cases you've never seen?** The design assumption is that we
*haven't* seen them all. So the safety is structural, not a list of patches:
**every** motion is (1) bounded by a sim model checked against the real floor with
a margin, (2) torque-capped so contact is gentle, (3) monitored during motion, and
(4) followed by a validation that **refuses** anything inconsistent rather than
shipping it. An unanticipated failure lands as a refused gate with a best-guess
remedy and a kept-safe fallback — not as a damaged robot or a silently-wrong file.

**Q. (the brag) Why is the procedure shaped exactly like this?** Every guardrail
here is the scar tissue of a real hardware incident — eleven of them, logged as
L-001…L-011 in the incident ledger: a gripper
jammed into the floor (→ a-priori sim planning), a 130° error from an encoder seam
(→ wrap-aware math), a calibration that was self-consistent but mirror-flipped (→
a registration phase), a stop that wasn't a stop (→ scale validation), a wrong-end
fold no camera could see (→ measured end-labels), a torque-starved battery (→
voltage-aware diagnosis). The procedure is, in effect, its own regression suite
for the physics of getting a cheap arm to agree with its model.

**Q. The sim model is the SO-100, but my arm is an SO-101 — does that matter?** No, for calibration. The model under `assets/so_arm100/` is the SO-100, and this hardware is an SO-101 (12 V / 30 kg·cm servos vs the SO-100's 7.4 V / 19.5 kg·cm). But the SO-100/101 **kinematic structure is identical**, and the redesigned SO-101 jaw has the **same reach** (model 10.2 cm from the wrist-roll axis vs ~10 cm measured), so the geometry the planner relies on is correct. The SO-101 changes are torque/voltage, wiring, and assembly — none geometric. The one surface-specific knob is `LEKIWI_FINGER_MARGIN` (default 1.5 cm), which accounts for **floor covering** standing above the wheel-contact plane the floor is derived from — raise it on soft flooring, set it to 0 on a hard floor.

## Why a JAX-native model (the part that isn't about calibration)

Be honest first: **you do not need JAX to calibrate this arm.** The offset/sign/
range map is arithmetic, and the LeRobot file it writes works with the stock
stack. If all you want is calibration, take the `out/` files and go.

So why is this built on a JAX-native physics model (the *jax\*y* stack —
[jaxonomy](https://github.com) engine → jaxterity robotics → jaxility deploy)?
Because calibration is the **foundation** everything else builds on. The hard part of robotics
isn't reading an encoder — it's having a *model of your specific robot that agrees
with reality*. This procedure produces exactly that as a byproduct: a model
**registered** to your hardware's frames and **validated** against its real stops.
Once that trustworthy model exists, everything downstream is differentiable,
batched, and deployable from the same object:

- **System identification instead of guessing.** The fidelity gaps we hit here —
  the model's jaw opening too little, link masses being nominal, the floor being a
  guess — are exactly what a differentiable model *fits*. Feed in a few seconds of
  telemetry and `grad` the simulated trajectory against the real one to recover
  masses, friction, and actuator constants. The hand-tuned constants in this
  notebook (sag budget, torque margins) become *learned* numbers.

- **A real-time digital twin.** With the map registered, live `PRESENT_POSITION`
  telemetry drives a faithful MuJoCo-MJX mirror at 30–60 Hz. The "verify the path,
  not just the endpoint" gap (L-010) closes completely: the twin watches the whole
  trajectory and predicts a floor/obstacle contact *before* it happens.

- **Controllers, synthesized not tuned.** The same calibrated model hands directly
  to LQR / MPC / whole-body control. `jit` for real-time rates, `grad` through the
  dynamics for trajectory optimization, all on the model you just proved correct.

- **`vmap` across a fleet.** Calibrate, plan, and tune *N* robots in parallel by
  vectorizing over instances — the per-instance offsets differ, the code doesn't.
  The thing that's irreducibly one-robot-at-a-time (each encoder's zero) is the one
  cheap arithmetic step; everything expensive is batched.

- **Autodiff the procedure itself.** The choreography's clearance margins, the
  stance search, the gravity-aware torque caps — today they're constants chosen by
  hand. In a differentiable simulator they're optimization variables: minimize
  calibration time subject to a hard floor-clearance constraint, by gradient.

- **One model, all the way to silicon.** The calibrated robot lowers (via jaxility)
  to a self-contained C project for an Arm SoC with a signed attestation manifest —
  the *same* model you simulated, calibrated, and controlled, compiled to run on
  the robot. CPU, GPU, or TPU during development; embedded C at deploy.
